# 01 - Landing to Bronze
Criação dos schemas, ingestão dos dados brutos da Landing Zone e extração da API do Banco Central, com adição do `ingestion_datetime` no momento da escrita.

In [0]:
# Parâmetros de padronização
catalog = "workspace"
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

landing_path = f"/Volumes/{catalog}/landing/arquivos_csv"

# Criação dos Schemas
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {bronze_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")

print("Schemas criados")

Schemas criados


In [0]:
# Validação da Landing Zone
import os

expected_files = [
    "credits_and_tags_IMDB_TMDB.csv",
    "movies_financials_IMDB_TMDB.csv",
    "movies_info_TMDB_IMDB.csv",
    "movies_metrics_IMDB_TMDB.csv",
    "movies_reviews.csv"
]

existing_files = os.listdir(landing_path)
missing_files = [f for f in expected_files if f not in existing_files]

if missing_files:
    print(f"ERRO: Os seguintes arquivos estão ausentes na Landing Zone: {missing_files}")
else:
    print("Todos os 5 arquivos esperados foram encontrados no Volume da Landing Zone!")

Todos os 5 arquivos esperados foram encontrados no Volume da Landing Zone!


In [0]:
# Mapeamento dos caminhos dos arquivos CSV na Landing Zone
path_movies_info = f"{landing_path}/movies_info_TMDB_IMDB.csv"
path_movies_financials = f"{landing_path}/movies_financials_IMDB_TMDB.csv"
path_movies_metrics = f"{landing_path}/movies_metrics_IMDB_TMDB.csv"
path_credits_and_tags = f"{landing_path}/credits_and_tags_IMDB_TMDB.csv"
path_movies_reviews = f"{landing_path}/movies_reviews.csv"

# Leitura dos CSVs com options de segurança para não ter quebras de linhas e aspas internas
df_movies_info_raw = spark.read.csv(path_movies_info, header=True, inferSchema=True, escape='"', multiLine=True)
df_movies_financials_raw = spark.read.csv(path_movies_financials, header=True, inferSchema=True, escape='"', multiLine=True)
df_movies_metrics_raw = spark.read.csv(path_movies_metrics, header=True, inferSchema=True, escape='"', multiLine=True)
df_credits_and_tags_raw = spark.read.csv(path_credits_and_tags, header=True, inferSchema=True, escape='"', multiLine=True)
df_movies_reviews_raw = spark.read.csv(path_movies_reviews, header=True, inferSchema=True, escape='"', multiLine=True)

In [0]:
from pyspark.sql.functions import current_timestamp

# Adição de ingestion_datetime no momento da escrita

# tb_movies_info
(
    df_movies_info_raw
    .withColumn("ingestion_datetime", current_timestamp())
    .write
    .format("delta")
    .mode("append")
    .saveAsTable(f"{bronze_schema}.tb_movies_info")
)

# tb_movies_financials
(
    df_movies_financials_raw
    .withColumn("ingestion_datetime", current_timestamp())
    .write
    .format("delta")
    .mode("append")
    .saveAsTable(f"{bronze_schema}.tb_movies_financials")
)

# tb_movies_metrics
(
    df_movies_metrics_raw
    .withColumn("ingestion_datetime", current_timestamp())
    .write
    .format("delta")
    .mode("append")
    .saveAsTable(f"{bronze_schema}.tb_movies_metrics")
)

# tb_credits_and_tags
(
    df_credits_and_tags_raw
    .withColumn("ingestion_datetime", current_timestamp())
    .write
    .format("delta")
    .mode("append")
    .saveAsTable(f"{bronze_schema}.tb_credits_and_tags")
)

# tb_movies_reviews
(
    df_movies_reviews_raw
    .withColumn("ingestion_datetime", current_timestamp())
    .write
    .format("delta")
    .mode("append")
    .saveAsTable(f"{bronze_schema}.tb_movies_reviews")
)

print("Todas as tabelas CSV foram gravadas na camada Bronze")

Todas as tabelas CSV foram gravadas na camada Bronze


In [0]:
import datetime
import requests
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

# Configuração dos Widgets para datas de início e fim
dbutils.widgets.text("data_inicio", "")
dbutils.widgets.text("data_fim", "")

input_inicio = dbutils.widgets.get("data_inicio")
input_fim = dbutils.widgets.get("data_fim")

# Se os widgets estiverem vazios calcula o intervalo automático dos últimos 7 dias
if not input_inicio or not input_fim:
    today = datetime.date.today()
    sete_dias_atras = today - datetime.timedelta(days=7)
    data_inicio_formatada = sete_dias_atras.strftime("%m-%d-%Y")
    data_fim_formatada = today.strftime("%m-%d-%Y")
else:
    data_inicio_formatada = input_inicio
    data_fim_formatada = input_fim

# Extração via API do Banco Central
url = f"https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)?@dataInicial='{data_inicio_formatada}'&@dataFinalCotacao='{data_fim_formatada}'&$select=dataHoraCotacao,cotacaoCompra&$format=json"

response = requests.get(url)

if response.status_code == 200:
    json_data = response.json().get("value", [])
    
    if json_data:
        # Schema explícito para parsing do JSON bruto
        api_schema = StructType([
            StructField("dataHoraCotacao", StringType(), True),
            StructField("cotacaoCompra", DoubleType(), True)
        ])
        
        df_api_raw = spark.createDataFrame(json_data, schema=api_schema)
        
        # Gravação na tabela bronze.tb_cotacao_dolar
        (
            df_api_raw
            .withColumn("ingestion_datetime", current_timestamp())
            .write
            .format("delta")
            .mode("append")
            .saveAsTable(f"{bronze_schema}.tb_cotacao_dolar")
        )
        print("Tabela bronze.tb_cotacao_dolar gravada com sucesso.")
    else:
        print("A API do BACEN retornou sem dados para o período informado.")
else:
    raise Exception(f"Falha na requisição da API BACEN. HTTP {response.status_code}")

Tabela bronze.tb_cotacao_dolar gravada com sucesso.


In [0]:
# Print de 5 exemplares para testar
display(spark.table(f"{bronze_schema}.tb_movies_info").limit(5))
display(spark.table(f"{bronze_schema}.tb_cotacao_dolar").limit(5))

id,tconst,title,original_title,original_language,release_date,runtime,status,overview,tagline,ingestion_datetime
293660,tt1431045,Deadpool,Deadpool,en,2016-02-09,108,Released,"The origin story of former Special Forces operative turned mercenary Wade Wilson, who, after being subjected to a rogue experiment that leaves him with accelerated healing powers, adopts the alter ego Deadpool. Armed with his new abilities and a dark, twisted sense of humor, Deadpool hunts down the man who nearly destroyed his life.",Witness the beginning of a happy ending.,2026-09-20T12:24:02.104Z
299536,tt4154756,AVENGERS: INFINITY WAR,Avengers: Infinity War,en,04-25-2018,149,Released,"As the Avengers and their allies have continued to protect the world from threats too large for any one hero to handle, a new danger has emerged from the cosmic shadows: Thanos. A despot of intergalactic infamy, his goal is to collect all six Infinity Stones, artifacts of unimaginable power, and use them to inflict his twisted will on all of reality. Everything the Avengers have fought for has led up to this moment - the fate of Earth and existence itself has never been more uncertain.",An entire universe. Once and for all.,2026-09-20T12:24:02.104Z
299534,tt4154796,Avengers: Endgame,Avengers: Endgame,en,2019-04-24,181,released,"After the devastating events of Avengers: Infinity War, the universe is in ruins due to the efforts of the Mad Titan, Thanos. With the help of remaining allies, the Avengers must assemble once more in order to undo Thanos' actions and restore order to the universe once and for all, no matter what consequences may be in store.",Avenge the fallen.,2026-09-20T12:24:02.104Z
475557,tt7286456,Joker,Joker,en,2019-10-01,122,Released,"During the 1980s, a failed stand-up comedian is driven insane and turns to a life of crime and chaos in Gotham City while becoming an infamous psychopathic crime figure.",Put on a happy face.,2026-09-20T12:24:02.104Z
271110,tt3498820,Captain America: Civil War,Captain America: Civil War,en,2016-04-27,147,Released,"Following the events of Age of Ultron, the collective governments of the world pass an act designed to regulate all superhuman activity. This polarizes opinion amongst the Avengers, causing two factions to side with Iron Man or Captain America, which causes an epic battle between former allies.",United we stand. Divided we fall.,2026-09-20T12:24:02.104Z


dataHoraCotacao,cotacaoCompra,ingestion_datetime
2026-09-14 13:10:08.144425,5.169,2026-09-20T12:24:22.552Z
2026-09-15 13:09:19.199664,5.1484,2026-09-20T12:24:22.552Z
2026-09-16 13:05:30.35873,5.152,2026-09-20T12:24:22.552Z
2026-09-17 13:03:21.858212,5.1515,2026-09-20T12:24:22.552Z
2026-09-18 13:03:34.742036,5.1569,2026-09-20T12:24:22.552Z
